# TEMP: Template Sensitivity of Matching Metrics

Scratch notebook (not part of the pipeline). Some expert phrasings are reused across many screens. An expert comment
is a **template** when its normalized `observed_issue` text occurs at least `TEMPLATE_MIN` times in the semantic-dedup
reference. Two questions:

1. Do models cover template comments more or less often than other expert comments?
2. Do the 2_b alignment metrics and the model ranking change when templates are excluded?

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "paths.py").exists())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
from utils.comments.pairs import explode_comments
from utils.paths import HUMAN_REFERENCE_DIR, RESULTS_DIR

MATCHING_DIR = RESULTS_DIR / "critiques" / "matching"
RUN_LABEL = "zero_shot-all_tasks"
MODEL_DISPLAY = {"claude4_7": "Claude Opus 4.7", "gpt5": "GPT-5", "gemini3_1pro": "Gemini 3.1 Pro"}
EXCLUDED_SCREEN_TASK_IDS = {"20234_T03", "31563_T02"}
RELATION_WEIGHTS = {
    "equivalent": 1.00, "B_broader": 0.85, "A_broader": 0.85, "partial_overlap": 0.50,
    "contradiction": 0.00, "related_but_different": 0.00, "no_match": 0.00,
}
TEMPLATE_MIN = 10
QUOTES = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})


def normalize_text(value):
    text = re.sub(r"\s+", " ", value.strip().lower().translate(QUOTES))
    return re.sub(r"^[^\w]+|[^\w]+$", "", text)

## 1. Load pairs and flag templates

In [2]:
keys = ["screen_task_id", "expert_comment_id", "llm_comment_id"]


def load_model(model):
    folder = MATCHING_DIR / model / RUN_LABEL
    stem = f"{RUN_LABEL}-{model}"
    labels = pd.read_parquet(folder / f"llmjudge_matching/llmjudge_matching_pairs-{stem}.parquet")[[*keys, "relation"]]
    texts = pd.read_parquet(folder / f"all_comment_pairs-{stem}.parquet")[[*keys, "app", "expert_observed_issue"]]
    return labels.merge(texts, on=keys, validate="one_to_one").assign(model=MODEL_DISPLAY[model])


pairs = pd.concat([load_model(m) for m in MODEL_DISPLAY], ignore_index=True)
pairs = pairs[~pairs["screen_task_id"].isin(EXCLUDED_SCREEN_TASK_IDS)]
pairs["rel_w"] = pairs["relation"].map(RELATION_WEIGHTS)

reference = explode_comments(pd.read_parquet(HUMAN_REFERENCE_DIR / "human_comments_semantic_dedup.parquet"), "comments")
template_freq = reference["observed_issue"].map(normalize_text).value_counts()
pairs["template"] = pairs["expert_observed_issue"].map(normalize_text).map(template_freq).ge(TEMPLATE_MIN)

expert_flags = pairs.drop_duplicates(["model", "screen_task_id", "expert_comment_id"])
expert_flags.groupby("model")["template"].agg(expert_comments="size", templates="sum", template_share="mean")

,expert_comments,templates,template_share
model,,,
Claude Opus 4.7,10488,2046,0.19508
GPT-5,10488,2046,0.19508
Gemini 3.1 Pro,10488,2046,0.19508


## 2. Template vs. other expert comments

Per expert comment, its relation weight in the bipartite assignment of 2_b (one LLM comment per expert comment).
Coverage is pooled per group; the gap is template minus other, with an app-clustered bootstrap 95% CI.

In [3]:
def bipartite_matches(df):
    rows = []
    for (model, task, app), g in df.groupby(["model", "screen_task_id", "app"], sort=False):
        rel = g.pivot_table(index="expert_comment_id", columns="llm_comment_id", values="rel_w", aggfunc="max", fill_value=0.0)
        r, c = linear_sum_assignment(-rel.to_numpy())
        matched = np.zeros(len(rel))
        matched[r] = rel.to_numpy()[r, c]
        rows.append(pd.DataFrame({"model": model, "screen_task_id": task, "app": app,
                                  "expert_comment_id": rel.index, "matched": matched, "llm_count": rel.shape[1]}))
    return pd.concat(rows, ignore_index=True)


matches = bipartite_matches(pairs).merge(
    expert_flags[["model", "screen_task_id", "expert_comment_id", "template"]], validate="one_to_one"
)


def coverage_gap(model_df, iterations=1000, seed=42):
    by_app = model_df.pivot_table(index="app", columns="template", values="matched", aggfunc=["sum", "count"], fill_value=0)
    gap = lambda s, n: s[True] / n[True] - s[False] / n[False]
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(iterations):
        sample = by_app.iloc[rng.integers(0, len(by_app), len(by_app))].sum()
        boot.append(gap(sample["sum"], sample["count"]))
    total = by_app.sum()
    return pd.Series({
        "template_cov": total["sum"][True] / total["count"][True],
        "other_cov": total["sum"][False] / total["count"][False],
        "gap": gap(total["sum"], total["count"]),
        "ci_low": np.percentile(boot, 2.5),
        "ci_high": np.percentile(boot, 97.5),
    })


matches.groupby("model").apply(coverage_gap).mul(100).round(1)

,template_cov,other_cov,gap,ci_low,ci_high
model,,,,,
Claude Opus 4.7,16.1,22.9,-6.8,-8.6,-5.1
GPT-5,14.2,20.8,-6.6,-8.1,-5.0
Gemini 3.1 Pro,11.6,18.0,-6.4,-7.9,-4.9


## 3. Metrics with templates excluded

Macro-averaged bipartite expert coverage and reference support as in 2_b section 2.1, once on all expert comments and
once after removing template comments (screens left without expert comments drop out).

In [4]:
def macro_metrics(df):
    per_screen = bipartite_matches(df).groupby(["model", "screen_task_id"]).agg(
        matched=("matched", "sum"), experts=("matched", "size"), llm=("llm_count", "first")
    )
    return pd.DataFrame({
        "expert_coverage": per_screen["matched"] / per_screen["experts"],
        "reference_support": per_screen["matched"] / per_screen["llm"],
    }).groupby("model").mean().mul(100)


comparison = pd.concat(
    {"all": macro_metrics(pairs), "templates excluded": macro_metrics(pairs[~pairs["template"]])}, axis=1
)
comparison[("change", "expert_coverage")] = comparison[("templates excluded", "expert_coverage")] - comparison[("all", "expert_coverage")]
comparison[("change", "reference_support")] = comparison[("templates excluded", "reference_support")] - comparison[("all", "reference_support")]
comparison.sort_values(("all", "expert_coverage"), ascending=False).round(1)

all                   templates excluded  \
                expert_coverage reference_support    expert_coverage   
model                                                                  
Claude Opus 4.7            23.6              11.4               24.4   
GPT-5                      21.3              11.8               22.5   
Gemini 3.1 Pro             18.9              18.2               19.8   

                                           change                    
                reference_support expert_coverage reference_support  
model                                                                
Claude Opus 4.7               9.9             0.8              -1.5  
GPT-5                        10.4             1.1              -1.4  
Gemini 3.1 Pro               16.1             0.9              -2.1